In [6]:
import requests
import pandas as pd
import time

In [16]:
# =========================
# API CONFIG
# =========================
Census_API_KEY = "407372f970401ad76187cf86e251a969bb40965d"
YEAR = 2022

Census_BASE_URL = f"https://api.census.gov/data/{YEAR}/acs/acs5"

params = {
    "get": "NAME,B25077_001E,B19013_001E,B01003_001E",
    "for": "county:*",
    "key": Census_API_KEY
}

# =========================
# API REQUEST
# =========================
response = requests.get(Census_BASE_URL, params=params)

print("Status Code:", response.status_code)

if response.status_code != 200:
    print(response.text)
    raise Exception("API request failed")

data = response.json()

# =========================
# RAW → DataFrame
# =========================
df_raw = pd.DataFrame(data[1:], columns=data[0])

df_raw.to_csv(f"census_raw_{YEAR}.csv", index=False)
print("Raw data saved.")

# =========================
# CLEANING
# =========================

df = df_raw.rename(columns={
    "NAME": "county_name",
    "B25077_001E": "median_home_value",
    "B19013_001E": "median_income",
    "B01003_001E": "population",
    "state": "state_fips",
    "county": "county_fips"
})

# FIPS → State Name
fips_to_state = {
    '01': 'Alabama','02': 'Alaska','04': 'Arizona','05': 'Arkansas',
    '06': 'California','08': 'Colorado','09': 'Connecticut',
    '10': 'Delaware','11': 'District of Columbia','12': 'Florida',
    '13': 'Georgia','15': 'Hawaii','16': 'Idaho','17': 'Illinois',
    '18': 'Indiana','19': 'Iowa','20': 'Kansas','21': 'Kentucky',
    '22': 'Louisiana','23': 'Maine','24': 'Maryland','25': 'Massachusetts',
    '26': 'Michigan','27': 'Minnesota','28': 'Mississippi',
    '29': 'Missouri','30': 'Montana','31': 'Nebraska','32': 'Nevada',
    '33': 'New Hampshire','34': 'New Jersey','35': 'New Mexico',
    '36': 'New York','37': 'North Carolina','38': 'North Dakota',
    '39': 'Ohio','40': 'Oklahoma','41': 'Oregon','42': 'Pennsylvania',
    '44': 'Rhode Island','45': 'South Carolina','46': 'South Dakota',
    '47': 'Tennessee','48': 'Texas','49': 'Utah','50': 'Vermont',
    '51': 'Virginia','53': 'Washington','54': 'West Virginia',
    '55': 'Wisconsin','56': 'Wyoming'
}

df["State_Name"] = df["state_fips"].map(fips_to_state)

# =========================
# TYPE CONVERSION
# =========================
df["median_home_value"] = pd.to_numeric(df["median_home_value"], errors="coerce")
df["median_income"] = pd.to_numeric(df["median_income"], errors="coerce")
df["population"] = pd.to_numeric(df["population"], errors="coerce")

# =========================
# REMOVE CENSUS INVALID VALUES
# =========================
# Census uses huge negative numbers for missing data
df = df[
    (df["median_home_value"] > 0) &
    (df["median_income"] > 0) &
    (df["population"] > 0)
]

# =========================
# FEATURE ENGINEERING
# =========================
df["price_to_income_ratio"] = df["median_home_value"] / df["median_income"]

# =========================
# STATE LEVEL AGGREGATION
# =========================

df_state = df.groupby("State_Name").agg({
    "median_home_value": "mean",
    "median_income": "mean",
    "population": "sum",
    "price_to_income_ratio": "mean"
}).reset_index()

# =========================
# SAVE
# =========================
df_state.to_csv(f"census_state_cleaned_{YEAR}.csv", index=False)

print("Cleaned data saved successfully!")
print(df_state.head())

Status Code: 200
Raw data saved.
Cleaned data saved successfully!
   State_Name  median_home_value  median_income  population  \
0     Alabama      140207.462687   51690.194030     5028092   
1      Alaska      252356.666667   76687.500000      734821   
2     Arizona      217293.333333   59725.533333     7172282   
3    Arkansas      124300.000000   49542.333333     3018669   
4  California      534350.000000   82966.603448    39356104   

   price_to_income_ratio  
0               2.683893  
1               3.229082  
2               3.567881  
3               2.492439  
4               6.096510  


In [12]:
# =====================================
# CONFIG
# =====================================

EPA_EMAIL = "y_lee49@u.pacific.edu"
EPA_KEY = "orangeram71"
YEAR = 2022

BASE_URL = "https://aqs.epa.gov/data/api/annualData/byState"

STATE_FIPS = [
    "01","02","04","05","06","08","09","10","11","12",
    "13","15","16","17","18","19","20","21","22","23",
    "24","25","26","27","28","29","30","31","32","33",
    "34","35","36","37","38","39","40","41","42","44",
    "45","46","47","48","49","50","51","53","54","55","56"
]

all_data = []

# =====================================
# Fetch Data (with retry + timeout)
# =====================================

def fetch_state_data(state, retries=3):
    params = {
        "email": EPA_EMAIL,
        "key": EPA_KEY,
        "param": "88101",
        "bdate": f"{YEAR}0101",
        "edate": f"{YEAR}1231",
        "state": state
    }

    for i in range(retries):
        try:
            response = requests.get(BASE_URL, params=params, timeout=10)

            if response.status_code == 200:
                return response.json()

            elif response.status_code == 429:
                print(f"Rate limit for {state}, retrying...")
                time.sleep(2)

        except requests.exceptions.RequestException:
            print(f"Error for state {state}, retry {i+1}")
            time.sleep(1)

    return None


# =====================================
# Main Loop
# =====================================

for state in STATE_FIPS:

    print(f"Fetching state {state}")

    json_data = fetch_state_data(state)

    if json_data is None or "Data" not in json_data:
        print(f"Skipping state {state}")
        continue

    df_state = pd.DataFrame(json_data["Data"])
    all_data.append(df_state)

    time.sleep(0.1)

# =====================================
# Combine
# =====================================

df_pm25 = pd.concat(all_data, ignore_index=True)

df_pm25 = df_pm25[[
    "state_code",
    "county_code",
    "arithmetic_mean"
]]

df_pm25["arithmetic_mean"] = pd.to_numeric(
    df_pm25["arithmetic_mean"], errors="coerce"
)

df_pm25 = df_pm25.dropna(subset=["arithmetic_mean"])

# =====================================
# State Aggregation
# =====================================

df_state_pm25 = (
    df_pm25
    .groupby("state_code", as_index=False)["arithmetic_mean"]
    .mean()
)

df_state_pm25 = df_state_pm25.rename(columns={
    "state_code": "state_fips",
    "arithmetic_mean": "pm25_state_mean"
})

# =====================================
# FIPS → State Name
# =====================================

fips_to_state = {
    '01': 'Alabama', '02': 'Alaska', '04': 'Arizona', '05': 'Arkansas',
    '06': 'California', '08': 'Colorado', '09': 'Connecticut',
    '10': 'Delaware', '11': 'District of Columbia', '12': 'Florida',
    '13': 'Georgia', '15': 'Hawaii', '16': 'Idaho', '17': 'Illinois',
    '18': 'Indiana', '19': 'Iowa', '20': 'Kansas', '21': 'Kentucky',
    '22': 'Louisiana', '23': 'Maine', '24': 'Maryland', '25': 'Massachusetts',
    '26': 'Michigan', '27': 'Minnesota', '28': 'Mississippi',
    '29': 'Missouri', '30': 'Montana', '31': 'Nebraska', '32': 'Nevada',
    '33': 'New Hampshire', '34': 'New Jersey', '35': 'New Mexico',
    '36': 'New York', '37': 'North Carolina', '38': 'North Dakota',
    '39': 'Ohio', '40': 'Oklahoma', '41': 'Oregon', '42': 'Pennsylvania',
    '44': 'Rhode Island', '45': 'South Carolina', '46': 'South Dakota',
    '47': 'Tennessee', '48': 'Texas', '49': 'Utah', '50': 'Vermont',
    '51': 'Virginia', '53': 'Washington', '54': 'West Virginia',
    '55': 'Wisconsin', '56': 'Wyoming'
}

df_state_pm25["State_Name"] = df_state_pm25["state_fips"].map(fips_to_state)

# =====================================
# Final Clean
# =====================================

df_state_pm25 = df_state_pm25[[
    "State_Name",
    "pm25_state_mean"
]]

df_state_pm25 = df_state_pm25.sort_values("State_Name").reset_index(drop=True)

# =====================================
# Validation
# =====================================

print("\n=== CHECK DATA ===")
print("Total states:", len(df_state_pm25))
print("Missing state names:", df_state_pm25["State_Name"].isna().sum())

# =====================================
# Save
# =====================================

df_state_pm25.to_csv(f"pm25_state_level_{YEAR}.csv", index=False)

print("\nState-level PM2.5 dataset saved successfully.")
print(df_state_pm25.head())

Fetching state 01
Fetching state 02
Fetching state 04
Fetching state 05
Fetching state 06
Fetching state 08
Fetching state 09
Fetching state 10
Fetching state 11
Fetching state 12
Fetching state 13
Fetching state 15
Fetching state 16
Fetching state 17
Fetching state 18
Fetching state 19
Fetching state 20
Fetching state 21
Fetching state 22
Fetching state 23
Fetching state 24
Fetching state 25
Fetching state 26
Fetching state 27
Fetching state 28
Fetching state 29
Fetching state 30
Fetching state 31
Fetching state 32
Fetching state 33
Fetching state 34
Fetching state 35
Fetching state 36
Fetching state 37
Fetching state 38
Fetching state 39
Fetching state 40
Fetching state 41
Fetching state 42
Fetching state 44
Fetching state 45
Fetching state 46
Fetching state 47
Fetching state 48
Fetching state 49
Fetching state 50
Fetching state 51
Fetching state 53
Fetching state 54
Fetching state 55
Fetching state 56

=== CHECK DATA ===
Total states: 51
Missing state names: 0

State-level PM2.5 dat

In [10]:
# FEMA Data Getting and Cleaning (FINAL VERSION)

import pandas as pd
import requests

START_YEAR = 2018
END_YEAR = 2022

BASE_URL = "https://www.fema.gov/api/open/v2/DisasterDeclarationsSummaries"

CLIMATE_TYPES = [
    "Hurricane",
    "Severe Storm",
    "Flood",
    "Fire",
    "Coastal Storm",
    "Tropical Storm",
    "Tornado"
]

all_data = []

# ==========================================
# Fetch FEMA Data with Pagination
# ==========================================

for year in range(START_YEAR, END_YEAR + 1):

    print(f"\nFetching FEMA data for {year}")

    skip = 0
    page_size = 1000

    while True:

        filter_query = (
            f"declarationDate ge '{year}-01-01T00:00:00.000Z' "
            f"and declarationDate le '{year}-12-31T23:59:59.999Z'"
        )

        params = {
            "$filter": filter_query,
            "$top": page_size,
            "$skip": skip
        }

        response = requests.get(BASE_URL, params=params)

        if response.status_code != 200:
            print(f"Failed request for year {year}")
            break

        data = response.json()

        if "DisasterDeclarationsSummaries" not in data:
            break

        df_page = pd.DataFrame(data["DisasterDeclarationsSummaries"])

        if df_page.empty:
            break

        all_data.append(df_page)

        skip += page_size
        print(f"Collected {skip} rows for {year}")

    print(f"Finished year {year}")

if len(all_data) == 0:
    raise ValueError("No FEMA data collected.")

# ==========================================
# Combine All Years
# ==========================================

df_disaster = pd.concat(all_data, ignore_index=True)

# Keep only climate-related disaster types
df_disaster = df_disaster[df_disaster["incidentType"].isin(CLIMATE_TYPES)]

# ==========================================
# Create State FIPS
# ==========================================

df_disaster["state_fips"] = df_disaster["fipsStateCode"].astype(str).str.zfill(2)

# ==========================================
# Aggregate to State Level (by disaster type)
# ==========================================

df_state_climate = (
    df_disaster
    .groupby(["state_fips", "incidentType"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

# ==========================================
# Rename Columns
# ==========================================

df_state_climate = df_state_climate.rename(columns={
    "Hurricane": "hurricane_count_5yr",
    "Severe Storm": "severe_storm_count_5yr",
    "Flood": "flood_count_5yr",
    "Fire": "fire_count_5yr",
    "Coastal Storm": "coastal_storm_count_5yr",
    "Tropical Storm": "tropical_storm_count_5yr",
    "Tornado": "tornado_count_5yr"
})

# ==========================================
# Create Total Count
# ==========================================

climate_columns = [
    "hurricane_count_5yr",
    "severe_storm_count_5yr",
    "flood_count_5yr",
    "fire_count_5yr",
    "coastal_storm_count_5yr",
    "tropical_storm_count_5yr",
    "tornado_count_5yr"
]

df_state_climate["climate_event_count_5yr"] = df_state_climate[climate_columns].sum(axis=1)

# ==========================================
# FIPS → State Name
# ==========================================

fips_to_state = {
    '01': 'Alabama', '02': 'Alaska', '04': 'Arizona', '05': 'Arkansas',
    '06': 'California', '08': 'Colorado', '09': 'Connecticut',
    '10': 'Delaware', '11': 'District of Columbia', '12': 'Florida',
    '13': 'Georgia', '15': 'Hawaii', '16': 'Idaho', '17': 'Illinois',
    '18': 'Indiana', '19': 'Iowa', '20': 'Kansas', '21': 'Kentucky',
    '22': 'Louisiana', '23': 'Maine', '24': 'Maryland', '25': 'Massachusetts',
    '26': 'Michigan', '27': 'Minnesota', '28': 'Mississippi',
    '29': 'Missouri', '30': 'Montana', '31': 'Nebraska', '32': 'Nevada',
    '33': 'New Hampshire', '34': 'New Jersey', '35': 'New Mexico',
    '36': 'New York', '37': 'North Carolina', '38': 'North Dakota',
    '39': 'Ohio', '40': 'Oklahoma', '41': 'Oregon', '42': 'Pennsylvania',
    '44': 'Rhode Island', '45': 'South Carolina', '46': 'South Dakota',
    '47': 'Tennessee', '48': 'Texas', '49': 'Utah', '50': 'Vermont',
    '51': 'Virginia', '53': 'Washington', '54': 'West Virginia',
    '55': 'Wisconsin', '56': 'Wyoming'
}

df_state_climate["State_Name"] = df_state_climate["state_fips"].map(fips_to_state)

# ==========================================
# 🚨 REMOVE INVALID STATES (VERY IMPORTANT)
# ==========================================

df_state_climate = df_state_climate.dropna(subset=["State_Name"])

# ==========================================
# Clean Final Columns
# ==========================================

df_state_climate = df_state_climate[[
    "State_Name",
    "hurricane_count_5yr",
    "severe_storm_count_5yr",
    "flood_count_5yr",
    "fire_count_5yr",
    "coastal_storm_count_5yr",
    "tropical_storm_count_5yr",
    "tornado_count_5yr",
    "climate_event_count_5yr"
]]

# Sort (optional but nice)
df_state_climate = df_state_climate.sort_values("State_Name")

# ==========================================
# Save Final Dataset
# ==========================================

df_state_climate.to_csv("fema_state_climate_5yr.csv", index=False)

print("\nClimate dataset saved successfully.")
print("Total states:", len(df_state_climate))
print(df_state_climate.head())


Fetching FEMA data for 2018
Collected 1000 rows for 2018
Collected 2000 rows for 2018
Finished year 2018

Fetching FEMA data for 2019
Collected 1000 rows for 2019
Collected 2000 rows for 2019
Finished year 2019

Fetching FEMA data for 2020
Collected 1000 rows for 2020
Collected 2000 rows for 2020
Collected 3000 rows for 2020
Collected 4000 rows for 2020
Collected 5000 rows for 2020
Collected 6000 rows for 2020
Collected 7000 rows for 2020
Collected 8000 rows for 2020
Collected 9000 rows for 2020
Collected 10000 rows for 2020
Finished year 2020

Fetching FEMA data for 2021
Collected 1000 rows for 2021
Collected 2000 rows for 2021
Finished year 2021

Fetching FEMA data for 2022
Collected 1000 rows for 2022
Collected 2000 rows for 2022
Finished year 2022

Climate dataset saved successfully.
Total states: 50
incidentType  State_Name  hurricane_count_5yr  severe_storm_count_5yr  \
0                Alabama                  127                      62   
1                 Alaska             

In [17]:
# =========================
# 🔹 1. 讀取資料
# =========================
home_df = pd.read_csv("cleaned_home_insurance_2022.csv")
crime_df = pd.read_csv("cleaned_crime_data.csv")
pm_df = pd.read_csv("pm25_state_level_2022.csv")
fema_df = pd.read_csv("fema_state_climate_5yr.csv")
census_df = pd.read_csv("census_state_cleaned_2022.csv")


# =========================
# 🔹 2. 統一 State_Name function
# =========================
def clean_state(df, col):
    df.rename(columns={col: "State_Name"}, inplace=True)
    df["State_Name"] = df["State_Name"].astype(str).str.strip().str.title()
    return df


# =========================
# 🔹 3. 清理每個 dataset
# =========================

# 保險
home_df = clean_state(home_df, "state")

# 犯罪（已經處理過）
crime_df = clean_state(crime_df, "State_Name")

# PM2.5
pm_df = clean_state(pm_df, "State_Name")

# FEMA
fema_df = clean_state(fema_df, "State_Name")

# Census
census_df = clean_state(census_df, "State_Name")


# =========================
# 🔹 4. Rename 指標欄位（讓 dashboard 好用）
# =========================

# 👉 這裡你可以依實際欄位調整
home_df.rename(columns={"avg_premium": "Insurance_Cost"}, inplace=True)

pm_df.rename(columns={"pm25": "PM25"}, inplace=True)

fema_df.rename(columns={"risk_score": "Climate_Risk"}, inplace=True)

census_df.rename(columns={"population": "Population"}, inplace=True)


# =========================
# 🔹 5. Merge（核心）
# =========================
merged_df = home_df.merge(crime_df, on="State_Name", how="inner")

merged_df = merged_df.merge(pm_df, on="State_Name", how="inner")

merged_df = merged_df.merge(fema_df, on="State_Name", how="inner")

merged_df = merged_df.merge(census_df, on="State_Name", how="inner")


# =========================
# 🔹 6. 檢查
# =========================
print("Final States:", merged_df['State_Name'].nunique())
print(merged_df.head())


# =========================
# 🔹 7. 存檔
# =========================
merged_df.to_csv("final_dataset.csv", index=False)

print("✅ Final dataset ready!")

Final States: 50
   State_Name  Avg_Home_Insurance_2022  Crime_Rate  pm25_state_mean  \
0     Alabama                  1414.11       443.7         8.031646   
1      Alaska                  1334.10         NaN         9.095974   
2     Arizona                  1085.10         NaN         7.593230   
3    Arkansas                  1273.60       648.0         8.503406   
4  California                  1619.62       531.3         8.805779   

   hurricane_count_5yr  severe_storm_count_5yr  flood_count_5yr  \
0                  127                      62                0   
1                    0                      10                4   
2                    0                       3                5   
3                   75                       0               36   
4                    0                      32                0   

   fire_count_5yr  coastal_storm_count_5yr  tropical_storm_count_5yr  \
0               0                        0                         0   
1        